# Notebook 04 — Tank-Level Model Training & Evaluation
## Machine Learning for Product Gain/Loss Prediction
**Author:** Uzoma Nnaemeka Eze | MSc Data Science | UE Potsdam

---
**Dataset:** D2 individual tank daily reports — 4,264 tank-day records (Feb–Dec 2025)  
**Products:** AGO (tanks T-101, T-102, T-121, T-122, T-141, T-142, T-5802, T-5803)  
              PMS (tanks T-123, T-124, T-143, T-144, T-5801, T-5804)  
**Key additional feature:** Opening_Dip_mm — physical tank level in millimetres  

**Key operational insight from this notebook:**
The depot hierarchy flows: 5800 series (primary storage) → 140 series (day tanks) → 100 series (dispatch)  
Large volume drops in 5800 series tanks are inter-tank transfers, NOT losses.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              roc_auc_score, classification_report, confusion_matrix,
                              roc_curve)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    tank_df = pd.read_csv('data/tank_feat.csv', parse_dates=['Date'])
    print(f"Tank data loaded: {len(tank_df)} records")
    print(f"AGO: {(tank_df['Product']=='AGO').sum()} | PMS: {(tank_df['Product']=='PMS').sum()}")
except:
    print("Run Notebook 02 first to generate tank_feat.csv")
    tank_df = pd.DataFrame()

TANK_FEATURES = ['Opening_Dip_mm','Water_Dip_mm','Temperature',
                 'Opening_Vol','Receipt','Dispatch','Has_Water',
                 'Dip_Change','Lag1_LG','Lag2_LG','Rolling3_LG',
                 'Has_Receipt','Has_Dispatch','DayOfWeek','Month_Num']

TANK_LABELS = ['Opening Dip (mm)','Water Dip (mm)','Temperature (C)',
               'Opening Volume','Receipt','Dispatch','Has Water',
               'Dip Change','Lag-1 L/G','Lag-2 L/G','Rolling 3-Day L/G',
               'Has Receipt','Has Dispatch','Day of Week','Month']


## 1. Train and Evaluate Tank-Level Models

In [ ]:
def train_evaluate_tank(product, color):
    sub = tank_df[tank_df['Product']==product].copy()
    sub = sub.sort_values('Date').reset_index(drop=True)
    sub = sub.dropna(subset=TANK_FEATURES+['Loss_Gain','Loss_Flag'])

    n = len(sub); split = int(n*0.80)
    print(f"\n{'='*60}")
    print(f"  {product} — Tank Level | 80/20 Chronological Split")
    print(f"{'='*60}")
    print(f"Records: {n} | Train: {split} | Test: {n-split}")
    print(f"Loss events: {sub['Loss_Flag'].sum()} ({sub['Loss_Flag'].mean()*100:.1f}%)")

    X = sub[TANK_FEATURES]; y_reg = sub['Loss_Gain']; y_cls = sub['Loss_Flag']
    X_tr,X_te = X.iloc[:split],X.iloc[split:]
    yr_tr,yr_te = y_reg.iloc[:split],y_reg.iloc[split:]
    yc_tr,yc_te = y_cls.iloc[:split],y_cls.iloc[split:]

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr); X_te_sc = sc.transform(X_te)

    lr = LinearRegression().fit(X_tr,yr_tr); lr_p = lr.predict(X_te)
    dtr = DecisionTreeRegressor(max_depth=6,random_state=42).fit(X_tr,yr_tr); dtr_p = dtr.predict(X_te)
    log = LogisticRegression(class_weight='balanced',random_state=42,max_iter=1000).fit(X_tr_sc,yc_tr)
    log_p = log.predict(X_te_sc); log_prob = log.predict_proba(X_te_sc)[:,1]
    dtc = DecisionTreeClassifier(max_depth=6,class_weight='balanced',random_state=42).fit(X_tr,yc_tr)
    dtc_p = dtc.predict(X_te); dtc_prob = dtc.predict_proba(X_te)[:,1]

    fi = pd.Series(dtr.feature_importances_, index=TANK_LABELS).sort_values(ascending=False)
    cm_log = confusion_matrix(yc_te,log_p)
    cm_dtc = confusion_matrix(yc_te,dtc_p)

    print(f"\nLinear Regression: MAE={mean_absolute_error(yr_te,lr_p):,.0f} L | R2={r2_score(yr_te,lr_p):.4f}")
    print(f"Decision Tree Reg: MAE={mean_absolute_error(yr_te,dtr_p):,.0f} L | R2={r2_score(yr_te,dtr_p):.4f}")
    print(f"Logistic Regression: AUC={roc_auc_score(yc_te,log_prob):.4f}")
    print(classification_report(yc_te,log_p,target_names=['Gain/Neutral','Loss'],digits=3))
    print(f"Decision Tree Classifier: AUC={roc_auc_score(yc_te,dtc_prob):.4f}")
    print(classification_report(yc_te,dtc_p,target_names=['Gain/Neutral','Loss'],digits=3))

    return dict(sub=sub, yr_te=yr_te, yc_te=yc_te, feat_imp=fi,
                log_prob=log_prob, log_auc=roc_auc_score(yc_te,log_prob), cm_log=cm_log,
                dtc_prob=dtc_prob, dtc_auc=roc_auc_score(yc_te,dtc_prob), cm_dtc=cm_dtc)

tank_results = {}
if len(tank_df) > 0:
    tank_results['AGO'] = train_evaluate_tank('AGO', '#1A5F9E')
    tank_results['PMS'] = train_evaluate_tank('PMS', '#C94E2D')


## 2. Dip Threshold Analysis — Loss Probability by Tank Level

In [ ]:
if len(tank_df) > 0:
    bins = [0,1000,2000,3000,5000,7000,9000,15000]
    lbls = ['0-1K','1-2K','2-3K','3-5K','5-7K','7-9K','9K+']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Loss Probability by Opening Dip Level — AGO and PMS', fontsize=12, fontweight='bold')

    for ax, (product, color) in zip(axes, [('AGO','#1A5F9E'),('PMS','#C94E2D')]):
        sub = tank_df[(tank_df['Product']==product) & (tank_df['Opening_Dip_mm']>0)].copy()
        sub['Loss_Flag'] = (sub['Loss_Gain'] < 0).astype(int)
        sub['Band'] = pd.cut(sub['Opening_Dip_mm'], bins=bins, labels=lbls)
        tbl = sub.groupby('Band',observed=True).agg(
            Count=('Loss_Flag','count'),
            Loss_Pct=('Loss_Flag',lambda x: x.mean()*100)
        ).reset_index()

        bar_colors = ['#E74C3C' if p>20 else '#E67E22' if p>12 else '#27AE60'
                      for p in tbl['Loss_Pct']]
        bars = ax.bar(tbl['Band'], tbl['Loss_Pct'], color=bar_colors, alpha=0.85, edgecolor='white')
        for bar, (_, row) in zip(bars, tbl.iterrows()):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f'{row["Loss_Pct"]:.1f}%\n(n={row["Count"]})',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        ax.axhline(20, color='red', lw=1.5, linestyle='--', label='20% high risk')
        ax.axhline(12, color='orange', lw=1.5, linestyle='--', label='12% moderate')
        ax.set_title(f'{product} — Loss Probability by Dip Level', fontweight='bold', color=color)
        ax.set_xlabel('Opening Dip Range (mm)')
        ax.set_ylabel('Loss Probability (%)')
        ax.legend(fontsize=8)
        ax.set_ylim(0, max(tbl['Loss_Pct'].max()+8, 30))
        ax.grid(axis='y', alpha=0.3)
        ax.set_facecolor('#FAFAFA')

    plt.tight_layout()
    plt.savefig('reports/fig_dip_threshold.png', dpi=150, bbox_inches='tight')
    plt.show()


## 3. Feature Importance — Tank-Level Decision Tree Regressor

In [ ]:
if tank_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Feature Importance — Tank-Level Decision Tree Regressor', fontsize=12, fontweight='bold')

    for ax, (product, color) in zip(axes, [('AGO','#1A5F9E'),('PMS','#C94E2D')]):
        fi = tank_results[product]['feat_imp'].sort_values(ascending=True)
        ax.barh(fi.index, fi.values, color=color, alpha=0.82, edgecolor='white')
        for i, (idx, val) in enumerate(fi.items()):
            ax.text(val+0.002, i, f'{val:.3f}', va='center', fontsize=8)
        ax.set_title(f'{product} — Feature Importance', fontweight='bold', color=color)
        ax.set_xlabel('Importance Score')
        ax.set_xlim(0, fi.max()*1.22)
        ax.grid(axis='x', alpha=0.25)
        ax.set_facecolor('#FAFAFA')
        ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('reports/fig_feature_importance_tank.png', dpi=150, bbox_inches='tight')
    plt.show()
